# EXP-014: Cân bằng lớp cho Random Forest

- Ngày: 22/11/2026 (tổng hợp notebook).
- Ngày chạy trong hồ sơ: 21/11/2026.
- Mã thực nghiệm: exp014.
- So với lần trước: exp009 (giữ tham số, thêm trọng số lớp).

**Thực nghiệm tương ứng:** [experiment.ipynb](../../experiments/03_balancing/EXP-014_RF/experiment.ipynb).

**Mạch thực nghiệm:** [EXP-005](Mai_EXP-005-RF.ipynb) → [EXP-009](Mai_EXP-009_RF_tuning.ipynb) → [EXP-014](Mai_EXP-014_RF_balancing.ipynb) → [EXP-019](Mai_EXP-019_RF_.ipynb).


## 1. Mục tiêu và giả thuyết

**Mục tiêu:** Kiểm tra riêng tác động của trọng số cân bằng lớp so với RF tuning chưa cân bằng.

**Giả thuyết:** Tăng trọng số cho lớp ít mẫu có thể cải thiện khả năng nhận diện lớp đó và các chỉ số trung bình theo lớp. Đây là kỳ vọng cần kiểm chứng.


## 2. Cấu hình

Giữ cấu hình tương ứng trong `experiment.ipynb` và hồ sơ lượt chạy.

Giữ `n_estimators`, `max_depth`, `max_features` đã được EXP-009 chọn theo đúng task và người test. Giữ đủ 68 biến, seed 42 và cách chia người.

**Tham số thay đổi:** `class_weight=None` → `class_weight="balanced"`. Không tìm lại tham số. Trọng số tính từ train, không tạo thêm mẫu.


## 3. Cách thực hiện

Dữ liệu physiological68 gồm 6.189 cửa sổ, 68 đặc trưng và 15 người. Nhị phân phân biệt không stress/stress; ba lớp gồm baseline/stress/amusement.

Lấy đúng tham số từng fold của EXP-009, thêm trọng số cân bằng rồi huấn luyện trên 14 người và kiểm tra người còn lại. Có 30 fold; so sánh cân bằng và chưa cân bằng trên cùng người test.

Mọi xử lý dữ liệu học từ train. Accuracy, Macro-F1 và Balanced Accuracy được tính cho từng người test rồi lấy trung bình đều qua 15 người.


## 4. Kết quả

Accuracy là tỷ lệ dự đoán đúng; Macro-F1 là trung bình F1 các lớp; Balanced Accuracy là trung bình recall các lớp. 

| Bài toán | Phiên bản | Accuracy | F1 (macro) | Balanced Accuracy |
|---|---|---|---|---|
| binary | rf | 0.8897 | 0.8657 | 0.8727 |
| multiclass | rf | 0.7668 | 0.6154 | 0.6526 |


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

here = Path.cwd()
repo = next(p for p in (here, *here.parents) if (p / 'src/wesad/evaluation.py').is_file())
source = repo / 'outputs/exp014_rf_mai_20261121'
file = source / ('results.csv' if (source / 'results.csv').exists() else 'metrics_by_subject.csv')
scores = pd.read_csv(file)
if 'model' in scores:
    scores = scores.loc[scores.model.eq('rf')]
if 'variant' not in scores:
    scores['variant'] = 'rf'
keys = ['task', 'variant']
display(scores.groupby(keys)[['macro_f1', 'balanced_accuracy']].agg(['mean', 'std']).round(4))


macro_f1         balanced_accuracy        
                       mean     std              mean     std
task       variant                                           
binary     rf        0.8657  0.1300            0.8727  0.1205
multiclass rf        0.6154  0.0957            0.6526  0.0817

### RF chưa cân bằng và cân bằng

![RF chưa cân bằng và cân bằng](../../experiments/03_balancing/EXP-014_RF/figures/rf_plain_vs_balanced.png)


### Train và test

![Train và test](../../experiments/03_balancing/EXP-014_RF/figures/train_vs_test.png)


### Nhầm lẫn giữa các lớp

![Nhầm lẫn giữa các lớp](../../experiments/03_balancing/EXP-014_RF/figures/rf_balancing_confusion_matrices.png)


## 5. So sánh với các lần trước

Các điểm đều là điểm test; chỉ so sánh trong cùng bài toán.

| Bài toán | Lần chạy | Thay đổi | Accuracy | F1 (macro) | Balanced Accuracy |
|---|---|---|---|---|---|
| binary | exp009 | RF tuning chưa cân bằng | 0.8834 | 0.8655 | 0.8782 |
| multiclass | exp009 | RF tuning chưa cân bằng | 0.7765 | 0.6540 | 0.6731 |
| binary | exp014 | Thêm trọng số cân bằng lớp | 0.8897 | 0.8657 | 0.8727 |
| multiclass | exp014 | Thêm trọng số cân bằng lớp | 0.7668 | 0.6154 | 0.6526 |


## 6. Nhận xét

- **Giả thuyết chưa được ủng hộ như cải thiện chung:** nhị phân tăng F1 khoảng 0,0002 nhưng giảm BA khoảng 0,0056. Ba lớp giảm F1 khoảng 0,0386 và BA khoảng 0,0206.
- **Quan sát bất thường:** thêm trọng số lớp chưa giúp cả hai bài toán cùng tốt hơn. Cần xem recall từng lớp và confusion matrix để biết lớp nào tăng, lớp nào giảm.
- **Lỗi / khó khăn:** phải ghép đúng tham số theo task và người test từ EXP-009. Điểm train được tính bổ sung sau khi đối chiếu dự đoán test. Khoảng cách train–test chưa xác định được một nguyên nhân duy nhất.


## 7. Bước tiếp theo

1. Chưa chọn cân bằng lớp như cải thiện chung cho cả hai bài toán.
2. Sang EXP-019 để thử lựa chọn đầu vào; dùng RF mặc định full68 làm đối chứng riêng.
3. Kiểm tra recall từng lớp và độ biến thiên theo người trước khi tổng hợp mô hình cuối.

**Liên kết:** [EXP-005](Mai_EXP-005-RF.ipynb) → [EXP-009](Mai_EXP-009_RF_tuning.ipynb) → [EXP-014](Mai_EXP-014_RF_balancing.ipynb) → [EXP-019](Mai_EXP-019_RF_.ipynb).
